# Notebook 156: 距離学習とファインチューニング

## Metric Learning & Fine-tuning

---

### このノートブックの位置づけ

**Embeddings シリーズ** の応用編として、埋め込み空間を「望ましい類似性」に合わせて変形する **距離学習（Metric Learning）** と、事前学習済みモデルのファインチューニングを学びます。

| 項目 | 内容 |
|------|------|
| 難易度 | ★★★★☆ |
| 所要時間 | 120〜150分 |
| カテゴリ | Embeddings |

### 学習目標

- [ ] 距離学習（Metric Learning）の目的と「何において似ているか」を制御する意義を説明できる
- [ ] Contrastive LossとTriplet Lossを数式から理解し、PyTorchで実装できる
- [ ] 効果的なペア/トリプレットのマイニング戦略を説明できる
- [ ] 事前学習済みモデルをドメイン特化タスクにファインチューニングできる
- [ ] 埋め込み空間の品質を定量評価できる

### 前提知識

- ✅ Notebook 150（埋め込みの幾何学）
- ✅ Notebook 140（表現学習 — 対照学習の基礎）
- ✅ PyTorch基礎（Notebook 35-36, 75-76）

---

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [なぜ距離学習が必要か？](#2-なぜ距離学習が必要か)
3. [Contrastive Loss](#3-contrastive-loss)
4. [Triplet Loss](#4-triplet-loss)
5. [FashionMNIST での距離学習実験](#5-fashionmnist-での距離学習実験)
6. [Hard Negative Mining の効果](#6-hard-negative-mining-の効果)
7. [文埋め込みのファインチューニング（概念）](#7-文埋め込みのファインチューニング概念)
8. [埋め込み品質の評価指標](#8-埋め込み品質の評価指標)
9. [まとめ・チートシート・よくある間違い・自己評価クイズ](#9-まとめチートシートよくある間違い自己評価クイズ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# Section 1: 環境セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 日本語フォント設定（環境に応じて調整）
# ------------------------------------------------------------
def setup_japanese_font():
    """日本語フォントをmatplotlibに設定する関数"""
    # macOS: Hiragino Sans, Windows: MS Gothic, Linux: IPAGothic
    font_candidates = ['Hiragino Sans', 'Arial Unicode MS', 'IPAGothic', 'MS Gothic', 'sans-serif']
    plt.rcParams['font.family'] = font_candidates
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (10, 6)
    plt.rcParams['font.size'] = 11
    print("✅ 日本語フォント設定完了")

setup_japanese_font()

# ------------------------------------------------------------
# 再現性のためのシード
# ------------------------------------------------------------
np.random.seed(42)
torch.manual_seed(42)

# Seaborn のスタイル設定
sns.set_style('whitegrid')

# ------------------------------------------------------------
# デバイス設定
# ------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ 環境セットアップ完了")
print(f"   NumPy version:   {np.__version__}")
print(f"   PyTorch version: {torch.__version__}")
print(f"   Device:          {device}")

---

## 2. なぜ距離学習が必要か？

### 2.1 汎用埋め込みの限界

事前学習済みモデルが出力する埋め込みは「汎用的」ですが、**全てのタスクに最適な埋め込みは存在しません**。

「類似性」はタスクに依存します。たとえば靴の画像を考えてみましょう:

- **色で似ている？** → 赤いスニーカーと赤いブーツが近い
- **形で似ている？** → 赤いスニーカーと青いスニーカーが近い
- **ブランドで似ている？** → Nikeのスニーカーと Nike のブーツが近い

### 2.2 距離学習の考え方

距離学習（Metric Learning）= 埋め込み空間を **「望ましい類似性」に合わせて変形する** 技術

```
【学習前: 汎用埋め込み空間】          【学習後: タスク特化埋め込み空間】

   ◆ △ ◆ ○                          ◆◆◆
  △  ○  △                           ◆ ◆
 ○ ◆  ○  △                              △△△
  △ ◆ ○ △                               △ △
   ○  ◆  ○                                   ○○○
                                              ○ ○
   混在した状態                     同クラスが近く、異クラスが遠い

   ◆ = クラスA, △ = クラスB, ○ = クラスC
```

### 2.3 距離学習の核心的アイデア

- **入力**: 「何と何が似ているか」の監督信号（ペアやトリプレット）
- **目的**: 似ているサンプルの埋め込みを近づけ、似ていないサンプルの埋め込みを遠ざける
- **出力**: タスクに最適化された埋め込み関数 $f_\theta(x)$

In [ ]:
# ============================================================
# Section 2: 距離学習の概念図
# ============================================================

# 学習前（混在）と学習後（分離）の埋め込み空間を可視化
np.random.seed(42)

# --- 学習前: ランダムに配置された3クラスのデータ ---
n_per_class = 30
# 混在した状態をシミュレート
before_class0 = np.random.randn(n_per_class, 2) * 1.5 + np.array([0.5, 0.5])
before_class1 = np.random.randn(n_per_class, 2) * 1.5 + np.array([-0.3, -0.2])
before_class2 = np.random.randn(n_per_class, 2) * 1.5 + np.array([0.1, -0.5])

# --- 学習後: クラスタが形成された状態 ---
after_class0 = np.random.randn(n_per_class, 2) * 0.3 + np.array([-3.0, 2.0])
after_class1 = np.random.randn(n_per_class, 2) * 0.3 + np.array([3.0, 2.0])
after_class2 = np.random.randn(n_per_class, 2) * 0.3 + np.array([0.0, -3.0])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 色とマーカーの設定
colors = ['#e74c3c', '#3498db', '#2ecc71']
labels = ['クラスA', 'クラスB', 'クラスC']
markers = ['o', 's', '^']

# --- 左: 学習前 ---
ax = axes[0]
for data, color, label, marker in zip(
    [before_class0, before_class1, before_class2],
    colors, labels, markers
):
    ax.scatter(data[:, 0], data[:, 1], c=color, label=label,
              marker=marker, s=60, alpha=0.7, edgecolors='black', linewidth=0.3)
ax.set_title('学習前: 汎用埋め込み空間\n（クラスが混在）', fontsize=13)
ax.set_xlabel('次元 1', fontsize=11)
ax.set_ylabel('次元 2', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# --- 右: 学習後 ---
ax = axes[1]
for data, color, label, marker in zip(
    [after_class0, after_class1, after_class2],
    colors, labels, markers
):
    ax.scatter(data[:, 0], data[:, 1], c=color, label=label,
              marker=marker, s=60, alpha=0.7, edgecolors='black', linewidth=0.3)
ax.set_title('学習後: 距離学習済み空間\n（同クラスが近く、異クラスが遠い）', fontsize=13)
ax.set_xlabel('次元 1', fontsize=11)
ax.set_ylabel('次元 2', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.suptitle('距離学習が埋め込み空間に与える効果', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("【ポイント】")
print("・学習前: 各クラスのサンプルが空間全体に散らばっている")
print("・学習後: 同クラスのサンプルが密集し、異クラスとは離れている")
print("・この変形を実現するのが Contrastive Loss や Triplet Loss")

---

## 3. Contrastive Loss

### 3.1 数式

Contrastive Loss は **ペア** を入力とする損失関数です。

$$
\mathcal{L} = (1 - y) \cdot \frac{1}{2} d^2 + y \cdot \frac{1}{2} \max(0, \text{margin} - d)^2
$$

ここで:
- $d = \|f(x_1) - f(x_2)\|_2$ : 埋め込み間のユークリッド距離
- $y = 0$ : **同クラスペア** (positive pair) → 距離 $d$ を小さくしたい
- $y = 1$ : **異クラスペア** (negative pair) → 距離 $d$ を margin 以上にしたい
- margin : 異クラスペアに要求する最小距離

### 3.2 直感的な理解

- **同クラスペア (y=0)**: 距離の2乗がそのままLossになる → 引き寄せる力
- **異クラスペア (y=1)**: 距離が margin 未満のときだけ Loss が発生 → 押し離す力
  - 距離が margin 以上なら Loss = 0（十分遠ければそれ以上は気にしない）

In [ ]:
# ============================================================
# Section 3: Contrastive Loss の実装と可視化
# ============================================================

class ContrastiveLoss(nn.Module):
    """
    Contrastive Loss の実装。
    
    ペア (x1, x2) と ラベル y を受け取り、
    同クラスペア (y=0) では距離を小さく、
    異クラスペア (y=1) では距離を margin 以上にする。
    
    Parameters:
        margin (float): 異クラスペアに要求する最小距離
    """
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
    
    def forward(self, embedding1, embedding2, label):
        """
        Parameters:
            embedding1: (batch_size, embed_dim) テンソル
            embedding2: (batch_size, embed_dim) テンソル
            label:      (batch_size,) テンソル (0=同クラス, 1=異クラス)
        Returns:
            loss: スカラーテンソル
        """
        # ユークリッド距離を計算
        distance = F.pairwise_distance(embedding1, embedding2, p=2)
        
        # 同クラスペア: 距離の2乗を小さくする
        positive_loss = (1 - label) * 0.5 * distance.pow(2)
        
        # 異クラスペア: 距離がmargin未満なら罰則
        negative_loss = label * 0.5 * F.relu(self.margin - distance).pow(2)
        
        # 平均を返す
        loss = (positive_loss + negative_loss).mean()
        return loss


# --- テスト ---
print("Contrastive Loss のテスト")
print("=" * 50)

criterion = ContrastiveLoss(margin=2.0)

# テスト: 同クラスペア（近い → Loss小）
emb1 = torch.tensor([[1.0, 0.0]])
emb2 = torch.tensor([[1.1, 0.1]])
label_pos = torch.tensor([0.0])  # 同クラス
loss_pos = criterion(emb1, emb2, label_pos)
print(f"同クラスペア（近い）: Loss = {loss_pos.item():.4f}")

# テスト: 同クラスペア（遠い → Loss大）
emb3 = torch.tensor([[1.0, 0.0]])
emb4 = torch.tensor([[-1.0, 2.0]])
loss_pos_far = criterion(emb3, emb4, label_pos)
print(f"同クラスペア（遠い）: Loss = {loss_pos_far.item():.4f}")

# テスト: 異クラスペア（遠い → Loss小）
label_neg = torch.tensor([1.0])  # 異クラス
emb5 = torch.tensor([[1.0, 0.0]])
emb6 = torch.tensor([[-2.0, 0.0]])  # 距離3.0 > margin=2.0
loss_neg_far = criterion(emb5, emb6, label_neg)
print(f"異クラスペア（遠い、d>margin）: Loss = {loss_neg_far.item():.4f}")

# テスト: 異クラスペア（近い → Loss大）
emb7 = torch.tensor([[1.0, 0.0]])
emb8 = torch.tensor([[1.5, 0.0]])  # 距離0.5 < margin=2.0
loss_neg_close = criterion(emb7, emb8, label_neg)
print(f"異クラスペア（近い、d<margin）: Loss = {loss_neg_close.item():.4f}")

In [ ]:
# ============================================================
# Plot 1: Contrastive Loss の形状を可視化
# ============================================================

# 距離 d の範囲を定義
d_range = np.linspace(0, 3.0, 300)

# 異なる margin 値で Loss を計算
margins = [0.5, 1.0, 1.5, 2.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左: 同クラスペア (y=0) ---
ax = axes[0]
# 同クラスペアの Loss は margin に依存しない
loss_positive = 0.5 * d_range ** 2
ax.plot(d_range, loss_positive, linewidth=2.5, color='#e74c3c', label='同クラスペア (y=0)')
ax.set_xlabel('距離 d', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Contrastive Loss: 同クラスペア\n（距離が大きいほど Loss が増加）', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 5)

# 矢印で「引き寄せる力」を示す
ax.annotate('距離を\n小さくしたい', xy=(1.5, 0.5*1.5**2), xytext=(2.2, 3.5),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5),
            fontsize=10, color='#e74c3c', ha='center')

# --- 右: 異クラスペア (y=1) ---
ax = axes[1]
cmap = plt.cm.viridis
for i, margin in enumerate(margins):
    # 異クラスペアの Loss
    loss_negative = 0.5 * np.maximum(0, margin - d_range) ** 2
    color = cmap(i / len(margins))
    ax.plot(d_range, loss_negative, linewidth=2.5, color=color,
            label=f'margin = {margin}')
    # margin の位置に縦線
    ax.axvline(x=margin, color=color, linestyle='--', alpha=0.4, linewidth=1)

ax.set_xlabel('距離 d', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Contrastive Loss: 異クラスペア\n（距離が margin 未満のときだけ Loss）', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 2.5)

# 矢印で「押し離す力」を示す
ax.annotate('d > margin なら\nLoss = 0', xy=(2.5, 0.0), xytext=(2.3, 1.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
            fontsize=10, color='gray', ha='center')

plt.suptitle('Plot 1: Contrastive Loss の形状', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("【ポイント】")
print("・同クラスペア: 距離に対してLossが二次的に増加（常にペナルティ）")
print("・異クラスペア: margin より遠ければペナルティなし（十分離れればOK）")
print("・margin が大きいほど、より強い分離を要求する")

---

## 4. Triplet Loss

### 4.1 数式

Triplet Loss は **3つ組（トリプレット）** を入力とする損失関数です。

$$
\mathcal{L} = \max\big(0, \; d(a, p) - d(a, n) + \text{margin}\big)
$$

ここで:
- $a$ : **anchor** （基準サンプル）
- $p$ : **positive** （anchor と同クラスのサンプル）
- $n$ : **negative** （anchor と異クラスのサンプル）
- $d(a, p)$ : anchor-positive 間の距離
- $d(a, n)$ : anchor-negative 間の距離
- margin : positive と negative の距離差に要求するマージン

### 4.2 直感的な理解

「anchor から positive までの距離」が「anchor から negative までの距離」より margin 以上小さくなることを要求する。

$$
d(a, p) + \text{margin} < d(a, n)
$$

In [ ]:
# ============================================================
# Section 4: Triplet Loss の実装と可視化
# ============================================================

class TripletLoss(nn.Module):
    """
    Triplet Loss の実装。
    
    anchor, positive, negative の3つ組を受け取り、
    anchor-positive 距離が anchor-negative 距離より
    margin 以上小さくなるよう学習する。
    
    Parameters:
        margin (float): 要求するマージン
    """
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin
    
    def forward(self, anchor, positive, negative):
        """
        Parameters:
            anchor:   (batch_size, embed_dim) テンソル
            positive: (batch_size, embed_dim) テンソル
            negative: (batch_size, embed_dim) テンソル
        Returns:
            loss: スカラーテンソル
        """
        # ユークリッド距離を計算
        d_ap = F.pairwise_distance(anchor, positive, p=2)
        d_an = F.pairwise_distance(anchor, negative, p=2)
        
        # Triplet Loss: max(0, d_ap - d_an + margin)
        loss = F.relu(d_ap - d_an + self.margin)
        
        return loss.mean()


# --- テスト ---
print("Triplet Loss のテスト")
print("=" * 50)

triplet_criterion = TripletLoss(margin=1.0)

# ケース1: 正しい配置（positive が近く、negative が遠い）
anchor = torch.tensor([[0.0, 0.0]])
pos = torch.tensor([[0.3, 0.3]])    # d_ap = 0.42
neg = torch.tensor([[3.0, 0.0]])    # d_an = 3.0
loss1 = triplet_criterion(anchor, pos, neg)
print(f"良い配置 (d_ap=0.42, d_an=3.0): Loss = {loss1.item():.4f}")

# ケース2: 悪い配置（negative が近い）
neg_close = torch.tensor([[0.5, 0.0]])  # d_an = 0.5
loss2 = triplet_criterion(anchor, pos, neg_close)
print(f"悪い配置 (d_ap=0.42, d_an=0.5): Loss = {loss2.item():.4f}")

# ケース3: margin 内（semi-hard negative）
neg_semi = torch.tensor([[1.0, 0.0]])  # d_an = 1.0
loss3 = triplet_criterion(anchor, pos, neg_semi)
print(f"Semi-hard  (d_ap=0.42, d_an=1.0): Loss = {loss3.item():.4f}")

In [ ]:
# ============================================================
# Plot 2: Triplet Loss の直感図
# anchor-positive 距離 vs anchor-negative 距離の2D空間
# ============================================================

margin = 1.0

# d_ap と d_an の範囲
d_ap_range = np.linspace(0, 3, 200)
d_an_range = np.linspace(0, 3, 200)
D_AP, D_AN = np.meshgrid(d_ap_range, d_an_range)

# Triplet Loss の値を計算
Loss_grid = np.maximum(0, D_AP - D_AN + margin)

fig, ax = plt.subplots(figsize=(8, 7))

# Loss のヒートマップ
contour = ax.contourf(D_AP, D_AN, Loss_grid, levels=20, cmap='YlOrRd')
plt.colorbar(contour, ax=ax, label='Triplet Loss')

# Loss = 0 の境界線（d_an = d_ap + margin）
ax.plot(d_ap_range, d_ap_range + margin, 'b--', linewidth=2.5,
        label=f'Loss = 0 の境界\n($d_{{an}} = d_{{ap}} + {margin}$)')

# d_ap = d_an の線
ax.plot(d_ap_range, d_ap_range, 'k:', linewidth=1.5,
        label='$d_{ap} = d_{an}$')

# 各領域にラベルを付ける
ax.text(0.5, 2.5, 'Loss = 0\n(十分に分離)', fontsize=12,
        ha='center', color='#2c3e50', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.text(2.2, 0.5, 'Loss > 0\n(改善が必要)', fontsize=12,
        ha='center', color='white', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#e74c3c', alpha=0.8))

# サンプルポイント
examples = [
    (0.5, 2.5, 'Easy negative', '#2ecc71'),
    (0.8, 1.5, 'Semi-hard negative', '#f39c12'),
    (1.5, 0.8, 'Hard negative', '#e74c3c'),
]
for d_ap_ex, d_an_ex, name, color in examples:
    ax.scatter(d_ap_ex, d_an_ex, c=color, s=150, zorder=5,
              edgecolors='black', linewidth=1.5)
    ax.annotate(name, (d_ap_ex, d_an_ex),
                textcoords='offset points', xytext=(10, -15),
                fontsize=10, fontweight='bold', color=color)

ax.set_xlabel('$d(a, p)$: anchor-positive 距離', fontsize=12)
ax.set_ylabel('$d(a, n)$: anchor-negative 距離', fontsize=12)
ax.set_title(f'Plot 2: Triplet Loss の直感図 (margin={margin})', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("【ポイント】")
print("・青い点線より上 (d_an > d_ap + margin): Loss = 0 → 学習不要")
print("・青い点線より下: Loss > 0 → negative をもっと遠ざけるか、positive を近づける必要")

In [ ]:
# ============================================================
# Triplet Mining 戦略の説明
# ============================================================

print("=" * 70)
print("Triplet Mining 戦略")
print("=" * 70)
print()
print("トリプレットの選び方が学習効率に大きく影響する")
print()
print("1. Easy Negatives (簡単な負例)")
print("   d(a,n) >> d(a,p) + margin → Loss = 0")
print("   学習に貢献しない（勾配 = 0）")
print()
print("2. Hard Negatives (困難な負例)")
print("   d(a,n) < d(a,p) → 最も情報量が多い")
print("   ただし、外れ値に引っ張られやすく不安定")
print()
print("3. Semi-Hard Negatives (半困難な負例)")
print("   d(a,p) < d(a,n) < d(a,p) + margin → 安定した学習")
print("   FaceNet 論文で推奨された戦略")
print()
print("""位置関係のイメージ:

  anchor(A)  positive(P)       margin 境界        negative の位置
     ●----------●                  |                     
     |  d(a,p)  |                  |                     
     |          |   d(a,p)+margin  |                     
     |          |         |        |                     
     ◄──────────►         |        |                     
                          |        |                     
          Hard ◆         Semi-Hard ◆         Easy ◆
       (d_an < d_ap)   (d_ap < d_an      (d_an >> d_ap 
                        < d_ap+m)          + margin)
""")
print("💡 実用上は Semi-Hard Mining が最もバランスが良い")

---

## 5. FashionMNIST での距離学習実験

FashionMNIST（10クラスのファッションアイテム画像）を使って、Triplet Loss による距離学習を実際に行います。

- 簡単な CNN エンコーダで **2D 埋め込み** に射影（可視化のため）
- バッチ内 Triplet Mining で効率的に学習
- 訓練前後の埋め込み空間を比較

In [ ]:
# ============================================================
# Section 5: FashionMNIST データのロードと前処理
# ============================================================

# FashionMNIST のクラス名（日本語）
class_names_jp = {
    0: 'Tシャツ',
    1: 'ズボン',
    2: 'プルオーバー',
    3: 'ドレス',
    4: 'コート',
    5: 'サンダル',
    6: 'シャツ',
    7: 'スニーカー',
    8: 'バッグ',
    9: 'アンクルブーツ',
}

# データの変換: テンソル化 + 正規化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# FashionMNIST のダウンロード・ロード
train_dataset_full = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset_full = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

# ------------------------------------------------------------
# CPU での実行速度のため、サブセットを使用
# 訓練: 各クラス500枚 = 5000枚
# テスト: 各クラス100枚 = 1000枚
# ------------------------------------------------------------
def get_balanced_subset(dataset, n_per_class=500):
    """
    データセットから各クラス均等にサンプリングする。
    
    Parameters:
        dataset: PyTorch Dataset
        n_per_class: 各クラスから取得するサンプル数
    Returns:
        Subset: バランスの取れたサブセット
    """
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset):
        class_indices[label].append(idx)
        # 十分なサンプルが集まったら早期終了
        if all(len(v) >= n_per_class for v in class_indices.values()):
            break
    
    selected_indices = []
    for label in sorted(class_indices.keys()):
        selected_indices.extend(class_indices[label][:n_per_class])
    
    return Subset(dataset, selected_indices)

train_dataset = get_balanced_subset(train_dataset_full, n_per_class=500)
test_dataset = get_balanced_subset(test_dataset_full, n_per_class=100)

print(f"✅ FashionMNIST ロード完了")
print(f"   訓練データ: {len(train_dataset)} 枚")
print(f"   テストデータ: {len(test_dataset)} 枚")
print(f"   クラス数: 10")
print(f"   画像サイズ: 28x28")

In [ ]:
# ============================================================
# CNN エンコーダの定義（2D 埋め込みに射影）
# ============================================================

class EmbeddingCNN(nn.Module):
    """
    FashionMNIST 画像を低次元の埋め込みに射影する CNN。
    
    アーキテクチャ:
        Conv2d(1, 32) -> ReLU -> MaxPool
        Conv2d(32, 64) -> ReLU -> MaxPool
        Flatten -> Linear(64*7*7, 128) -> ReLU
        Linear(128, embed_dim)
    
    Parameters:
        embed_dim (int): 埋め込み次元数（デフォルト: 2）
    """
    def __init__(self, embed_dim=2):
        super(EmbeddingCNN, self).__init__()
        # 畳み込み層
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # 全結合層
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, embed_dim)
    
    def forward(self, x):
        """
        Parameters:
            x: (batch_size, 1, 28, 28) テンソル
        Returns:
            embedding: (batch_size, embed_dim) テンソル
        """
        # 畳み込み + プーリング
        x = self.pool(F.relu(self.conv1(x)))   # -> (batch, 32, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))   # -> (batch, 64, 7, 7)
        
        # 平坦化
        x = x.view(x.size(0), -1)              # -> (batch, 64*7*7)
        
        # 全結合層
        x = F.relu(self.fc1(x))                # -> (batch, 128)
        embedding = self.fc2(x)                 # -> (batch, embed_dim)
        
        return embedding


# モデルのテスト
model_test = EmbeddingCNN(embed_dim=2).to(device)
dummy_input = torch.randn(4, 1, 28, 28).to(device)
dummy_output = model_test(dummy_input)
print(f"モデル出力の形状: {dummy_output.shape}")  # (4, 2)

# パラメータ数
total_params = sum(p.numel() for p in model_test.parameters())
print(f"総パラメータ数: {total_params:,}")
print("\n✅ EmbeddingCNN 定義完了")

In [ ]:
# ============================================================
# バッチ内 Triplet Mining の実装
# ============================================================

def batch_hard_triplet_loss(embeddings, labels, margin=1.0):
    """
    バッチ内 Hard Triplet Mining による Triplet Loss。
    
    各 anchor に対して:
    - hardest positive: 同クラスで最も遠いサンプル
    - hardest negative: 異クラスで最も近いサンプル
    
    Parameters:
        embeddings: (batch_size, embed_dim) テンソル
        labels:     (batch_size,) テンソル
        margin:     マージン値
    Returns:
        loss: スカラーテンソル
        n_valid: 有効なトリプレット数
    """
    # ペアワイズ距離行列を計算
    # ||a - b||^2 = ||a||^2 + ||b||^2 - 2 * a . b
    pairwise_dist = torch.cdist(embeddings, embeddings, p=2)
    
    # ラベルのマスクを作成
    labels_equal = labels.unsqueeze(0) == labels.unsqueeze(1)  # (B, B)
    labels_not_equal = ~labels_equal
    
    # --- Hardest Positive ---
    # 同クラスペアの距離を取得（異クラスは -inf に）
    positive_dist = pairwise_dist * labels_equal.float()
    # 各 anchor について最も遠い positive を選択
    hardest_positive_dist, _ = positive_dist.max(dim=1)
    
    # --- Hardest Negative ---
    # 異クラスペアの距離を取得（同クラスは大きな値に）
    large_val = pairwise_dist.max() + 1.0
    negative_dist = pairwise_dist + labels_equal.float() * large_val
    # 各 anchor について最も近い negative を選択
    hardest_negative_dist, _ = negative_dist.min(dim=1)
    
    # Triplet Loss
    triplet_loss = F.relu(hardest_positive_dist - hardest_negative_dist + margin)
    
    # 有効なトリプレット数（Loss > 0）
    n_valid = (triplet_loss > 0).sum().item()
    
    return triplet_loss.mean(), n_valid


def batch_semihard_triplet_loss(embeddings, labels, margin=1.0):
    """
    バッチ内 Semi-Hard Triplet Mining による Triplet Loss。
    
    Semi-hard negative: d(a,p) < d(a,n) < d(a,p) + margin
    
    Parameters:
        embeddings: (batch_size, embed_dim) テンソル
        labels:     (batch_size,) テンソル
        margin:     マージン値
    Returns:
        loss: スカラーテンソル
        n_valid: 有効なトリプレット数
    """
    pairwise_dist = torch.cdist(embeddings, embeddings, p=2)
    batch_size = embeddings.size(0)
    
    labels_equal = labels.unsqueeze(0) == labels.unsqueeze(1)
    labels_not_equal = ~labels_equal
    
    total_loss = torch.tensor(0.0, device=embeddings.device)
    n_valid = 0
    
    for i in range(batch_size):
        # anchor = i
        # positive: 同クラスの他のサンプル
        pos_mask = labels_equal[i].clone()
        pos_mask[i] = False  # 自分自身を除く
        
        if pos_mask.sum() == 0:
            continue
        
        # ランダムに positive を選択
        pos_indices = pos_mask.nonzero(as_tuple=True)[0]
        pos_idx = pos_indices[torch.randint(len(pos_indices), (1,))]
        d_ap = pairwise_dist[i, pos_idx]
        
        # Semi-hard negative: d_ap < d_an < d_ap + margin
        neg_mask = labels_not_equal[i]
        neg_dists = pairwise_dist[i]
        
        # Semi-hard 条件
        semi_hard_mask = neg_mask & (neg_dists > d_ap) & (neg_dists < d_ap + margin)
        
        if semi_hard_mask.sum() > 0:
            # Semi-hard negatives の中から最も近いものを選択
            semi_hard_dists = neg_dists.clone()
            semi_hard_dists[~semi_hard_mask] = float('inf')
            neg_idx = semi_hard_dists.argmin()
            d_an = pairwise_dist[i, neg_idx]
        elif neg_mask.sum() > 0:
            # Semi-hard がなければ Hard negative にフォールバック
            hard_dists = neg_dists.clone()
            hard_dists[~neg_mask] = float('inf')
            neg_idx = hard_dists.argmin()
            d_an = pairwise_dist[i, neg_idx]
        else:
            continue
        
        loss_i = F.relu(d_ap - d_an + margin)
        if loss_i > 0:
            n_valid += 1
        total_loss = total_loss + loss_i
    
    if batch_size > 0:
        total_loss = total_loss / batch_size
    
    return total_loss, n_valid


def batch_random_triplet_loss(embeddings, labels, margin=1.0):
    """
    バッチ内 Random Triplet Mining による Triplet Loss。
    positive と negative をランダムに選択する（ベースライン）。
    
    Parameters:
        embeddings: (batch_size, embed_dim) テンソル
        labels:     (batch_size,) テンソル
        margin:     マージン値
    Returns:
        loss: スカラーテンソル
        n_valid: 有効なトリプレット数
    """
    pairwise_dist = torch.cdist(embeddings, embeddings, p=2)
    batch_size = embeddings.size(0)
    
    labels_equal = labels.unsqueeze(0) == labels.unsqueeze(1)
    labels_not_equal = ~labels_equal
    
    total_loss = torch.tensor(0.0, device=embeddings.device)
    n_valid = 0
    
    for i in range(batch_size):
        # positive: 同クラスからランダム
        pos_mask = labels_equal[i].clone()
        pos_mask[i] = False
        if pos_mask.sum() == 0:
            continue
        pos_indices = pos_mask.nonzero(as_tuple=True)[0]
        pos_idx = pos_indices[torch.randint(len(pos_indices), (1,))]
        d_ap = pairwise_dist[i, pos_idx]
        
        # negative: 異クラスからランダム
        neg_mask = labels_not_equal[i]
        if neg_mask.sum() == 0:
            continue
        neg_indices = neg_mask.nonzero(as_tuple=True)[0]
        neg_idx = neg_indices[torch.randint(len(neg_indices), (1,))]
        d_an = pairwise_dist[i, neg_idx]
        
        loss_i = F.relu(d_ap - d_an + margin)
        if loss_i > 0:
            n_valid += 1
        total_loss = total_loss + loss_i
    
    if batch_size > 0:
        total_loss = total_loss / batch_size
    
    return total_loss, n_valid


print("✅ バッチ内 Triplet Mining 関数定義完了")
print("   - batch_hard_triplet_loss:     Hard Mining")
print("   - batch_semihard_triplet_loss:  Semi-Hard Mining")
print("   - batch_random_triplet_loss:    Random Mining (ベースライン)")

In [ ]:
# ============================================================
# 埋め込みの抽出・可視化ヘルパー関数
# ============================================================

def extract_embeddings(model, dataset, device, batch_size=256):
    """
    モデルを使って全データの埋め込みを抽出する。
    
    Parameters:
        model:    EmbeddingCNN モデル
        dataset:  PyTorch Dataset
        device:   torch.device
        batch_size: バッチサイズ
    Returns:
        embeddings: (N, embed_dim) numpy配列
        labels:     (N,) numpy配列
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    all_embeddings = []
    all_labels = []
    
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            emb = model(images)
            all_embeddings.append(emb.cpu().numpy())
            all_labels.append(targets.numpy())
    
    embeddings = np.concatenate(all_embeddings, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    
    return embeddings, labels


def plot_embeddings(embeddings, labels, title, ax=None, class_names=None):
    """
    2D 埋め込みの散布図を描画する。
    
    Parameters:
        embeddings: (N, 2) numpy配列
        labels:     (N,) numpy配列
        title:      図のタイトル
        ax:         matplotlib Axes (省略時は新規作成)
        class_names: ラベル→名前の辞書
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    # 10クラス用のカラーパレット
    palette = plt.cm.tab10(np.linspace(0, 1, 10))
    
    unique_labels = np.unique(labels)
    for label in unique_labels:
        mask = labels == label
        name = class_names[label] if class_names else str(label)
        ax.scatter(
            embeddings[mask, 0], embeddings[mask, 1],
            c=[palette[label]], s=15, alpha=0.6,
            label=name, edgecolors='none'
        )
    
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('埋め込み次元 1', fontsize=11)
    ax.set_ylabel('埋め込み次元 2', fontsize=11)
    ax.legend(fontsize=8, markerscale=3, loc='best', ncol=2)
    ax.grid(True, alpha=0.3)


print("✅ ヘルパー関数定義完了")

In [ ]:
# ============================================================
# Plot 3: 訓練前の埋め込み空間（ランダム初期化）
# ============================================================

# 新しいモデルを作成（ランダム初期化状態）
torch.manual_seed(42)
model_before = EmbeddingCNN(embed_dim=2).to(device)

# テストデータの埋め込みを抽出
embeddings_before, labels_before = extract_embeddings(
    model_before, test_dataset, device
)

fig, ax = plt.subplots(figsize=(10, 8))
plot_embeddings(
    embeddings_before, labels_before,
    title='Plot 3: 訓練前の埋め込み空間（ランダム初期化）\n→ クラスが混在している',
    ax=ax, class_names=class_names_jp
)
plt.tight_layout()
plt.show()

print("【観察】")
print("・ランダム初期化された CNN の埋め込みは構造を持たない")
print("・各クラスが混在しており、クラスタは見られない")
print("・これを距離学習で改善する")

In [ ]:
# ============================================================
# Triplet Loss による訓練ループ
# ============================================================

def train_metric_learning(model, dataset, loss_fn, n_epochs=20,
                          batch_size=128, lr=1e-3, margin=1.0,
                          device='cpu', verbose=True):
    """
    距離学習の訓練ループ。
    
    Parameters:
        model:      EmbeddingCNN モデル
        dataset:    訓練データセット
        loss_fn:    損失関数（batch_hard/semihard/random_triplet_loss）
        n_epochs:   エポック数
        batch_size: バッチサイズ
        lr:         学習率
        margin:     Triplet Loss のマージン
        device:     torch.device
        verbose:    進捗表示
    Returns:
        loss_history: エポックごとの平均 Loss のリスト
    """
    # DataLoader: バッチ内に複数クラスが含まれるようシャッフル
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        drop_last=True)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    
    model.train()
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        n_batches = 0
        total_valid_triplets = 0
        
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            # 順伝播
            embeddings = model(images)
            
            # Loss を計算
            loss, n_valid = loss_fn(embeddings, labels, margin=margin)
            
            # 逆伝播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            total_valid_triplets += n_valid
            n_batches += 1
        
        avg_loss = epoch_loss / max(n_batches, 1)
        loss_history.append(avg_loss)
        
        if verbose and (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:3d}/{n_epochs}: "
                  f"Loss = {avg_loss:.4f}, "
                  f"有効トリプレット = {total_valid_triplets}")
    
    return loss_history


# --- Hard Mining での訓練 ---
print("=" * 60)
print("Triplet Loss (Hard Mining) による訓練開始")
print("=" * 60)

torch.manual_seed(42)
np.random.seed(42)
model_hard = EmbeddingCNN(embed_dim=2).to(device)

loss_history_hard = train_metric_learning(
    model=model_hard,
    dataset=train_dataset,
    loss_fn=batch_hard_triplet_loss,
    n_epochs=20,
    batch_size=128,
    lr=1e-3,
    margin=1.0,
    device=device,
    verbose=True
)

print(f"\n✅ 訓練完了: 最終 Loss = {loss_history_hard[-1]:.4f}")

In [ ]:
# ============================================================
# Plot 4: 訓練後の埋め込み空間
# ============================================================

# 訓練後の埋め込みを抽出
embeddings_after, labels_after = extract_embeddings(
    model_hard, test_dataset, device
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- 左: 訓練前 ---
plot_embeddings(
    embeddings_before, labels_before,
    title='訓練前（ランダム初期化）',
    ax=axes[0], class_names=class_names_jp
)

# --- 右: 訓練後 ---
plot_embeddings(
    embeddings_after, labels_after,
    title='訓練後（Triplet Loss + Hard Mining, 20 epochs）',
    ax=axes[1], class_names=class_names_jp
)

plt.suptitle('Plot 4: FashionMNIST 埋め込み空間の訓練前後比較', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("【観察】")
print("・訓練後: 各クラスが明確なクラスタを形成")
print("・同クラスのサンプルが近くに集まり、異クラスとは離れている")
print("・Triplet Loss が埋め込み空間を効果的に構造化した")

In [ ]:
# ============================================================
# Plot 5: 学習曲線（Loss vs epoch）
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))

epochs = range(1, len(loss_history_hard) + 1)
ax.plot(epochs, loss_history_hard, 'o-', linewidth=2, markersize=6,
        color='#e74c3c', label='Hard Mining')

ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('平均 Triplet Loss', fontsize=12)
ax.set_title('Plot 5: 学習曲線（Triplet Loss + Hard Mining）', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【観察】")
print("・Loss が徐々に減少している")
print("・Hard Mining では常に困難なトリプレットを選ぶため、")
print("  Loss が 0 に近づきにくいが、学習は効率的に進む")

---

## 6. Hard Negative Mining の効果

3つのマイニング戦略（Random / Semi-Hard / Hard）を比較し、学習効率の違いを確認します。

In [ ]:
# ============================================================
# Section 6: 3つのマイニング戦略の比較実験
# ============================================================

# --- Random Mining ---
print("--- Random Mining ---")
torch.manual_seed(42)
np.random.seed(42)
model_random = EmbeddingCNN(embed_dim=2).to(device)
loss_history_random = train_metric_learning(
    model=model_random,
    dataset=train_dataset,
    loss_fn=batch_random_triplet_loss,
    n_epochs=20,
    batch_size=128,
    lr=1e-3,
    margin=1.0,
    device=device,
    verbose=False
)
print(f"  最終 Loss: {loss_history_random[-1]:.4f}")

# --- Semi-Hard Mining ---
print("--- Semi-Hard Mining ---")
torch.manual_seed(42)
np.random.seed(42)
model_semihard = EmbeddingCNN(embed_dim=2).to(device)
loss_history_semihard = train_metric_learning(
    model=model_semihard,
    dataset=train_dataset,
    loss_fn=batch_semihard_triplet_loss,
    n_epochs=20,
    batch_size=128,
    lr=1e-3,
    margin=1.0,
    device=device,
    verbose=False
)
print(f"  最終 Loss: {loss_history_semihard[-1]:.4f}")

# Hard Mining は既に訓練済み
print(f"--- Hard Mining (既訓練) ---")
print(f"  最終 Loss: {loss_history_hard[-1]:.4f}")

print("\n✅ 3つの戦略の訓練完了")

In [ ]:
# ============================================================
# Plot 6: 3つのマイニング戦略の Loss 曲線比較
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

epochs = range(1, 21)

ax.plot(epochs, loss_history_random, 's-', linewidth=2, markersize=5,
        color='#3498db', label='Random Mining', alpha=0.8)
ax.plot(epochs, loss_history_semihard, '^-', linewidth=2, markersize=5,
        color='#2ecc71', label='Semi-Hard Mining', alpha=0.8)
ax.plot(epochs, loss_history_hard, 'o-', linewidth=2, markersize=5,
        color='#e74c3c', label='Hard Mining', alpha=0.8)

ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('平均 Triplet Loss', fontsize=12)
ax.set_title('Plot 6: マイニング戦略による学習曲線の比較', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【観察】")
print("・Random Mining: Easy negative が多く含まれるため、Loss が速く下がるが")
print("  実際には学習が十分進んでいない可能性がある")
print("・Hard Mining: 常に困難なトリプレットを選ぶため Loss は高めだが、")
print("  埋め込み空間の品質は良い")
print("・Semi-Hard Mining: バランスの取れた学習で、安定した収束")

---

## 7. 文埋め込みのファインチューニング（概念）

### 7.1 なぜファインチューニングが必要か

事前学習済みの文埋め込みモデル（例: `sentence-transformers`）は汎用的ですが、ドメイン特化のタスクでは精度が不足することがあります。

| ドメイン | 課題の例 |
|---------|----------|
| 医療テキスト | 「頭痛」と「偏頭痛」の類似性を高くしたい |
| 法律文書 | 「契約違反」と「債務不履行」の類似性 |
| 技術文書 | 「メモリリーク」と「OOM エラー」の類似性 |

### 7.2 訓練データの形式

ファインチューニングに使うデータは主に3つの形式があります:

1. **トリプレット**: (anchor, positive, negative)
2. **ペア + スコア**: (sentence1, sentence2, similarity_score)
3. **ペア + ラベル**: (sentence1, sentence2, is_similar)

In [ ]:
# ============================================================
# Section 7: sentence-transformers ファインチューニングの概念コード
# ============================================================

# 注意: このセクションは概念的なコードです。
# 実際の大規模訓練は GPU と大量のデータを必要とします。

print("=" * 60)
print("sentence-transformers ファインチューニングの概要")
print("=" * 60)

print("""
【ステップ 1: モデルとデータの準備】

```python
from sentence_transformers import (
    SentenceTransformer, 
    InputExample, 
    losses
)
from torch.utils.data import DataLoader

# 事前学習済みモデルをロード
model = SentenceTransformer('all-MiniLM-L6-v2')

# 訓練データの準備（トリプレット形式）
train_examples = [
    InputExample(
        texts=['頭痛がひどい', '偏頭痛の症状', '胃が痛い']
    ),
    InputExample(
        texts=['契約違反について', '債務不履行の事例', '天気予報']
    ),
    # ... 数千〜数万のトリプレット
]

train_dataloader = DataLoader(
    train_examples, shuffle=True, batch_size=16
)
```

【ステップ 2: 損失関数の選択】

```python
# Triplet Loss を使用
train_loss = losses.TripletLoss(model=model)

# あるいは、ペア + スコア形式なら CosineSimilarityLoss
# train_loss = losses.CosineSimilarityLoss(model=model)
```

【ステップ 3: ファインチューニング実行】

```python
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    output_path='./finetuned_model'
)
```

【ステップ 4: ファインチューニング後の推論】

```python
# ドメイン特化の類似度がより正確に
sentences = ['頭痛がする', '偏頭痛の治療法', '腹痛の原因']
embeddings = model.encode(sentences)
# → '頭痛がする' と '偏頭痛の治療法' の類似度が
#   ファインチューニング前より高くなる
```
""")

print("💡 ファインチューニングの Tips:")
print("  1. 訓練データの質 > 量（ノイジーなデータは逆効果）")
print("  2. 学習率は小さめ（1e-5 ~ 5e-5）が安全")
print("  3. エポック数は 1-5 で十分なことが多い（過学習に注意）")
print("  4. 評価データセットで定期的に品質を確認")

---

## 8. 埋め込み品質の評価指標

距離学習の結果を定量的に評価するための指標を実装します。

| 指標 | 説明 |
|------|------|
| Recall@K | 上位 K 件の最近傍に同クラスのサンプルがあるか |
| Mean Average Precision (MAP) | 検索結果全体の精度の平均 |
| Normalized Mutual Information (NMI) | クラスタリング結果とラベルの一致度 |
| Silhouette Score | クラスタの分離度 |

In [ ]:
# ============================================================
# Section 8: 埋め込み品質の評価指標の実装
# ============================================================

def recall_at_k(embeddings, labels, k=5):
    """
    Recall@K: 各サンプルの上位K最近傍に同クラスのサンプルが
    少なくとも1つ含まれている割合。
    
    Parameters:
        embeddings: (N, D) numpy配列
        labels:     (N,) numpy配列
        k:          上位何件を見るか
    Returns:
        float: Recall@K スコア (0.0 ~ 1.0)
    """
    # k+1 で検索（自分自身を除くため）
    nn_model = NearestNeighbors(n_neighbors=k+1, metric='euclidean')
    nn_model.fit(embeddings)
    _, indices = nn_model.kneighbors(embeddings)
    
    # 自分自身を除いた上位K件
    neighbor_indices = indices[:, 1:k+1]
    
    # 各サンプルについて、上位K件に同クラスがあるかチェック
    hits = 0
    for i in range(len(labels)):
        neighbor_labels = labels[neighbor_indices[i]]
        if labels[i] in neighbor_labels:
            hits += 1
    
    return hits / len(labels)


def mean_average_precision(embeddings, labels, k=10):
    """
    Mean Average Precision (MAP@K):
    各クエリの Average Precision の平均。
    
    Parameters:
        embeddings: (N, D) numpy配列
        labels:     (N,) numpy配列
        k:          上位何件を見るか
    Returns:
        float: MAP@K スコア (0.0 ~ 1.0)
    """
    nn_model = NearestNeighbors(n_neighbors=k+1, metric='euclidean')
    nn_model.fit(embeddings)
    _, indices = nn_model.kneighbors(embeddings)
    
    # 自分自身を除く
    neighbor_indices = indices[:, 1:k+1]
    
    aps = []
    for i in range(len(labels)):
        query_label = labels[i]
        neighbor_labels = labels[neighbor_indices[i]]
        
        # Average Precision を計算
        n_relevant = 0
        precision_sum = 0.0
        for j, nl in enumerate(neighbor_labels):
            if nl == query_label:
                n_relevant += 1
                precision_sum += n_relevant / (j + 1)
        
        if n_relevant > 0:
            aps.append(precision_sum / min(n_relevant, k))
        else:
            aps.append(0.0)
    
    return np.mean(aps)


def compute_nmi(embeddings, labels, n_clusters=10):
    """
    Normalized Mutual Information (NMI):
    KMeans クラスタリング結果と真のラベルの一致度。
    
    Parameters:
        embeddings: (N, D) numpy配列
        labels:     (N,) numpy配列
        n_clusters: KMeans のクラスタ数
    Returns:
        float: NMI スコア (0.0 ~ 1.0)
    """
    from sklearn.metrics import normalized_mutual_info_score
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)
    return normalized_mutual_info_score(labels, cluster_labels)


def compute_silhouette(embeddings, labels):
    """
    Silhouette Score: クラスタの分離度を測る。
    +1 に近いほどクラスタが良く分離されている。
    
    Parameters:
        embeddings: (N, D) numpy配列
        labels:     (N,) numpy配列
    Returns:
        float: Silhouette Score (-1.0 ~ 1.0)
    """
    return silhouette_score(embeddings, labels)


print("✅ 評価指標の関数定義完了")
print("   - recall_at_k()")
print("   - mean_average_precision()")
print("   - compute_nmi()")
print("   - compute_silhouette()")

In [ ]:
# ============================================================
# 訓練前後の評価指標を計算
# ============================================================

print("=" * 60)
print("訓練前後の埋め込み品質を評価中...")
print("=" * 60)

# --- 訓練前 ---
print("\n--- 訓練前（ランダム初期化）---")
r1_before = recall_at_k(embeddings_before, labels_before, k=1)
r5_before = recall_at_k(embeddings_before, labels_before, k=5)
map_before = mean_average_precision(embeddings_before, labels_before, k=10)
nmi_before = compute_nmi(embeddings_before, labels_before)
sil_before = compute_silhouette(embeddings_before, labels_before)

print(f"  Recall@1:         {r1_before:.4f}")
print(f"  Recall@5:         {r5_before:.4f}")
print(f"  MAP@10:           {map_before:.4f}")
print(f"  NMI:              {nmi_before:.4f}")
print(f"  Silhouette Score: {sil_before:.4f}")

# --- 訓練後 ---
print("\n--- 訓練後（Hard Mining, 20 epochs）---")
r1_after = recall_at_k(embeddings_after, labels_after, k=1)
r5_after = recall_at_k(embeddings_after, labels_after, k=5)
map_after = mean_average_precision(embeddings_after, labels_after, k=10)
nmi_after = compute_nmi(embeddings_after, labels_after)
sil_after = compute_silhouette(embeddings_after, labels_after)

print(f"  Recall@1:         {r1_after:.4f}")
print(f"  Recall@5:         {r5_after:.4f}")
print(f"  MAP@10:           {map_after:.4f}")
print(f"  NMI:              {nmi_after:.4f}")
print(f"  Silhouette Score: {sil_after:.4f}")

In [ ]:
# ============================================================
# Plot 7: 評価指標の訓練前後比較（棒グラフ）
# ============================================================

metrics_names = ['Recall@1', 'Recall@5', 'MAP@10', 'NMI', 'Silhouette']
scores_before = [r1_before, r5_before, map_before, nmi_before, sil_before]
scores_after = [r1_after, r5_after, map_after, nmi_after, sil_after]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width/2, scores_before, width, label='訓練前',
               color='#bdc3c7', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, scores_after, width, label='訓練後',
               color='#e74c3c', edgecolor='black', linewidth=0.5)

# 値をバーの上に表示
for bar, val in zip(bars1, scores_before):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar, val in zip(bars2, scores_after):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10,
            fontweight='bold', color='#e74c3c')

ax.set_xlabel('評価指標', fontsize=12)
ax.set_ylabel('スコア', fontsize=12)
ax.set_title('Plot 7: 埋め込み品質の訓練前後比較', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

print("【結果の解釈】")
print("・全ての指標で訓練後が大幅に改善")
print("・Recall@1: 最近傍検索の精度が向上")
print("・NMI: クラスタリングとラベルの一致度が向上")
print("・Silhouette: クラスタの分離度が向上")
print("\n→ 距離学習が埋め込み空間の品質を大幅に改善した")

---

## 9. まとめ・チートシート・よくある間違い・自己評価クイズ

### 9.1 まとめ

このノートブックでは、距離学習（Metric Learning）とファインチューニングの理論と実装を学びました。

1. **距離学習の必要性**: 汎用埋め込みは全タスクに最適ではない → タスク特化の埋め込み空間を学習
2. **Contrastive Loss**: ペアベースの損失関数、同クラスを引き寄せ異クラスを押し離す
3. **Triplet Loss**: トリプレットベースの損失関数、相対的な距離関係を学習
4. **Mining 戦略**: Hard/Semi-Hard/Random の選択が学習効率に大きく影響
5. **FashionMNIST 実験**: 実際に Triplet Loss で埋め込み空間を構造化
6. **評価指標**: Recall@K, MAP, NMI, Silhouette Score で品質を定量評価

### 9.2 チートシート

| Loss | 入力形式 | 特徴 | 適用場面 |
|------|---------|------|----------|
| Contrastive Loss | ペア (x1, x2, y) | 絶対距離を制御、シンプル | 検証タスク（同一人物か？）|
| Triplet Loss | トリプレット (a, p, n) | 相対距離を制御、表現力高い | 検索・推薦・Re-ID |
| N-pair Loss | N個の負例 | 多数の負例を同時に活用 | 大規模検索 |
| InfoNCE / NT-Xent | バッチ内対照 | 自己教師あり学習にも使える | 対照学習 (SimCLR等) |

**Mining 戦略:**

| 戦略 | 特徴 | 推奨場面 |
|------|------|----------|
| Random | 簡単だが非効率 | ベースライン、デバッグ |
| Hard | 最も情報量が多い、不安定 | データがクリーンな場合 |
| Semi-Hard | バランスが良い、安定 | 一般的に推奨 |

### 9.3 よくある間違い

#### 間違い 1: Easy Negatives ばかりで Loss が 0 だが学習が進まない

Random Mining では、ほとんどのトリプレットが Easy Negative（Loss = 0）になりがちです。Loss が 0 に見えても、それは「学習が完了した」のではなく「有用なトリプレットが選ばれていない」だけです。**有効なトリプレット数**（Loss > 0 のトリプレット）を監視し、Hard または Semi-Hard Mining に切り替えましょう。

#### 間違い 2: margin の設定が不適切

- **margin が大きすぎ**: 収束しにくくなる（要求が厳しすぎて達成できない）
- **margin が小さすぎ**: クラス間の分離が不十分（簡単に条件を満たしてしまう）

一般的には margin = 0.2 ~ 2.0 の範囲で実験し、検証データの評価指標で選択します。

#### 間違い 3: 埋め込み次元を大きくしすぎる

「次元が高いほど表現力が高い」とは限りません。特に訓練データが少ない場合、高次元の埋め込みは過学習しやすくなります。**まず 2D で実験して可視化**し、動作を確認してから次元を拡大するのが良いプラクティスです。

### 9.4 自己評価クイズ

---

**Q1.** Contrastive Loss と Triplet Loss の入力形式の違いを説明してください。

<details>
<summary>回答を表示</summary>

- **Contrastive Loss**: **ペア** (x1, x2) とラベル y（同クラス=0 or 異クラス=1）を入力とする。絶対的な距離を制御する。
- **Triplet Loss**: **3つ組** (anchor, positive, negative) を入力とする。相対的な距離関係（positive がnegative より近いこと）を学習する。

Triplet Loss の方が自由度が高く、「どれくらい近いか」ではなく「相対的にどちらが近いか」を学習するため、多くの場合で性能が良い。

</details>

---

**Q2.** Semi-Hard Negative とは何ですか？なぜ Hard Negative より安定した学習ができるのですか？

<details>
<summary>回答を表示</summary>

Semi-Hard Negative は、$d(a,p) < d(a,n) < d(a,p) + \text{margin}$ を満たす負例です。つまり、positive より遠いが margin の範囲内にある負例です。

Hard Negative ($d(a,n) < d(a,p)$) は最も情報量が多いですが、外れ値やノイズの影響を受けやすく、勾配が大きくなりすぎて訓練が不安定になることがあります。Semi-Hard は「適度に難しい」トリプレットなので、安定した勾配で着実に学習が進みます。

</details>

---

**Q3.** Recall@1 が 0.8 で Recall@5 が 0.95 の場合、何が分かりますか？

<details>
<summary>回答を表示</summary>

- **Recall@1 = 0.8**: 最近傍のサンプルが同クラスである確率が 80%
- **Recall@5 = 0.95**: 上位5件のうち少なくとも1つが同クラスである確率が 95%

この差は、埋め込み空間のクラス境界付近に一部のサンプルが混在していることを示唆します。最近傍1件だけでは間違えることがあるが、5件まで見ればほぼ正解が含まれる、というのは実用上十分な品質です。

</details>

---

**Q4.** 事前学習済みの文埋め込みモデルをファインチューニングする際、学習率を小さくする（1e-5 程度）のはなぜですか？

<details>
<summary>回答を表示</summary>

事前学習済みモデルは、大規模データで学習した汎用的な知識を既に持っています。学習率が大きすぎると、この **事前学習の知識を破壊（catastrophic forgetting）** してしまいます。小さな学習率で「微調整」することで、汎用知識を保持しつつドメイン特化の類似性を学習できます。

</details>

---

**Q5.** 距離学習で margin を 0.5 から 2.0 に変更すると、埋め込み空間にどのような影響がありますか？

<details>
<summary>回答を表示</summary>

margin を大きくすると、異クラスのサンプル間により大きな距離を要求するため、クラス間の分離がより強くなります。ただし、以下のトレードオフがあります:

- **メリット**: クラス間の境界がより明確になり、検索精度が向上する可能性
- **デメリット**: 要求が厳しすぎると収束が遅くなったり、埋め込み空間が崩壊する可能性

適切な margin は データの性質とタスクに依存するため、検証データで実験的に決定するのが一般的です。

</details>

### 9.5 次のステップ

次の **Notebook 157: 埋め込みの応用と統合** では、ここで学んだ距離学習の技術を実際の応用に結びつけます。

- 類似画像検索システムの構築
- マルチモーダル埋め込み（画像 + テキスト）
- 埋め込みのインデックスと高速検索（FAISS）

---

## 参考文献

1. Hadsell, R., Chopra, S., & LeCun, Y. (2006). *Dimensionality Reduction by Learning an Invariant Mapping*. CVPR.
2. Schroff, F., Kalenichenko, D., & Philbin, J. (2015). *FaceNet: A Unified Embedding for Face Recognition and Clustering*. CVPR.
3. Hermans, A., Beyer, L., & Leibe, B. (2017). *In Defense of the Triplet Loss for Person Re-Identification*. arXiv:1703.07737.
4. Reimers, N., & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. EMNLP.
5. Musgrave, K., Belongie, S., & Lim, S.-N. (2020). *A Metric Learning Reality Check*. ECCV.